# Módulo 4 · Reto integrador: una ejecución que pueda explicarse

**Modalidad:** tarea extra autónoma · **Tiempo estimado:** 3 horas

## El problema
El equipo comercial necesita ejecutar periódicamente un informe de ventas por canal y rango de fechas. El análisis actual funciona en la computadora de quien lo creó, pero no permite reconstruir con seguridad **qué entorno se usó, qué datos se descartaron, con qué parámetros se ejecutó ni por qué cambió el resultado**.

Tu reto es transformar este notebook en un proceso **reproducible, observable y automatizable**.

## Resultado esperado
Una persona que no participó de tu trabajo debe poder:

1. recrear el entorno;
2. ejecutar este mismo notebook con parámetros diferentes;
3. leer el log y reconstruir las decisiones de limpieza;
4. localizar los artefactos de cada corrida;
5. verificar los resultados sin depender del orden manual de las celdas.

## Entrega

- este notebook completado y ejecutado de principio a fin;
- dos notebooks ejecutados con Papermill;
- para cada corrida: `detalle.csv`, `resumen.json`, `entorno.json` y `ejecucion.log`;
- matriz de contraste con IA y decisión técnica final de 250–350 palabras.


## Presupuesto de trabajo (180 minutos)

| Tramo | Tiempo orientativo |
|---|---:|
| Preparación y preflight | 15 min |
| Entorno y logger | 25 min |
| Pipeline y reglas de calidad | 40 min |
| Orquestación, artefactos y controles | 30 min |
| Dos corridas con Papermill | 20 min |
| Contraste con IA | 20 min |
| Decisión y revisión final | 10 min |
| Margen técnico y recreación limpia | 20 min |

**Camino mínimo de aprobación:** una solución correcta para la política de mediana, dos corridas, cuatro artefactos por `run_id`, controles superados, contraste crítico y decisión final. La extensión opcional no forma parte del mínimo.


In [ ]:
# Esta la agregue para que me muestre la version de python y el path del ejecutable y el prefix
import sys
print("Version:", sys.version.split()[0])
print("Executable:", sys.executable)
print("Prefix:", sys.prefix)

Version: 3.12.10
Executable: c:\codigo\python\ambientes\ProgramacionAvanzada_Reto-Modulo-4\.venv\Scripts\python.exe
Prefix: c:\codigo\python\ambientes\ProgramacionAvanzada_Reto-Modulo-4\.venv


## Reglas no negociables

- No uses rutas personales ni absolutas.
- El notebook debe superar **Reiniciar kernel y ejecutar todo**.
- La celda `parameters` es la única fuente de parámetros de negocio.
- Cada corrida usa un `run_id` diferente y conserva sus propios artefactos.
- El log debe incluir parámetros, conteos, advertencias y fallas; no debe volcar filas completas ni datos sensibles.
- No ocultes errores con `except: pass`: registralos y relanzalos cuando impidan continuar.
- Primero realizá tu diagnóstico humano. Recién después consultá IA.
- Si comparás modelos de IA, usá exactamente el mismo notebook, prompt y evidencia para ambos.

**Criterio de aprobación:** no alcanza con que “dé un número”. La ejecución debe dejar evidencia que permita explicar y repetir ese número.


## 0. Preparación reproducible

Desde una terminal ubicada en la carpeta del módulo:

**macOS / Linux**
```bash
## verificar version disponible 
python3.12 --version
python3.12 -m venv .venv
source .venv/bin/activate
python -m pip install -r requirements.txt
python -m ipykernel install --user --name unicaba-mod4 --display-name "UNICABA M4"
```

**Windows PowerShell**
```powershell
## verificar primero la version sea python 3.12
py -3.12 --version
py -3.12 -m venv .venv
.venv\Scripts\Activate.ps1
# Verificar la versión del intérprete
python --version
python -m pip install -r requirements.txt
python -m ipykernel install --user --name unicaba-mod4 --display-name "UNICABA M4"
```

Verificá con:

```bash
python -m pip check
python -c "import pandas, papermill; print('entorno listo')"
```

Seleccioná el kernel **UNICABA M4**. No entregues la carpeta `.venv`. Entregá el archivo de dependencias y el recibo de entorno generado por este notebook.

**Prueba de recreación:** al finalizar, recreá el entorno en una carpeta limpia (o pedí a otra persona que lo haga), ejecutá `python -m pip check` y compará `entorno.json` y `resumen.json` con tu corrida original.


## Hoja de ruta

`entorno + parámetros → logging → limpieza → artefactos → Papermill → contraste con IA → decisión`


## Una referencia técnica mínima

Una ruta relativa no significa “adivinar dónde está el archivo”. El patrón siguiente busca la carpeta del módulo por una evidencia estable: `Data/ventas_ejemplo.csv`. Ésta es la única resolución completa provista; el resto del pipeline es parte del reto.


In [3]:
from pathlib import Path

def encontrar_raiz_materiales() -> Path:
    for candidato in (Path.cwd(), *Path.cwd().parents):
        if (candidato / "Data" / "ventas_ejemplo.csv").exists():
            return candidato
    raise FileNotFoundError(
        "No se encontró Data/ventas_ejemplo.csv. Abrí Jupyter desde la carpeta del módulo."
    )

RAIZ = encontrar_raiz_materiales()
DATA_PATH = RAIZ / "Data" / "ventas_ejemplo.csv"
RAIZ


WindowsPath('c:/codigo/python/ambientes/ProgramacionAvanzada_Reto-Modulo-4')

## Hito 1 · Parámetros y trazabilidad

No renombres estas variables: Papermill las inyectará al ejecutar el notebook.


In [55]:
# Esta celda debe conservar la etiqueta `parameters`.
canal = "Web"
fecha_inicio = "2026-01-01"
fecha_fin = "2026-02-28"
run_id = "manual_web"
# run_id = "manual_web_revision_01"
directorio_salidas = "resultados/reto_integrador"


In [ ]:
# ### Si las fechas contienen un formato inválido, fromisoformat() genera un error. 
# ### Si están invertidas, falla nuestra aserción.

# # # from datetime import date

# inicio = date.fromisoformat(fecha_inicio)
# fin = date.fromisoformat(fecha_fin)

# assert inicio <= fin, "La fecha inicial no puede superar la fecha final"
# # assert fin <= inicio, "La fecha inicial no puede superar la fecha final"
# print("Período válido:", inicio, "hasta", fin)


## Hito 2 · Recibo de entorno y logger

Construí evidencia del entorno y un logger seguro para reejecutar en Jupyter.


In [ ]:
# TODO 1 · Completá los cuerpos; conservá estas firmas.
import hashlib
import json
import logging
import platform
import sys
from importlib.metadata import PackageNotFoundError, version

import pandas as pd
from pathlib import Path

def capturar_entorno(data_path: Path, requirements_path: Path) -> dict:
    """Captura las versiones del sistema y el hash (SHA-256) de los archivos."""
    #def obtener_hash(ruta: Path):
        #return hashlib.sha256(ruta.read_bytes()).hexdigest() if ruta.exists() else "No encontrado"
    def obtener_hash(ruta: Path) -> str:
        if not ruta.is_file():
            raise FileNotFoundError(f"No existe el archivo: {ruta}")

        return hashlib.sha256(ruta.read_bytes()).hexdigest()
    try:
        papermill_ver = version("papermill")
    except PackageNotFoundError:
        papermill_ver = "No instalado"

    return {
        "sistema_operativo": platform.platform(),
        "version_python": sys.version.split(" ")[0],
        "version_pandas": pd.__version__,
        "version_papermill": papermill_ver,
        "hash_datos_csv": obtener_hash(data_path),
        "hash_requirements": obtener_hash(requirements_path)
    }

def crear_logger(nombre: str, archivo: Path) -> logging.Logger:
    """Crea un logger seguro sin handlers duplicados (INFO en consola, DEBUG en archivo)."""
    logger = logging.getLogger(nombre)
    logger.setLevel(logging.DEBUG)

    # LIMPIEZA CLAVE: Evita handlers duplicados si ejecutamos la celda varias veces
    # if logger.hasHandlers():
        #logger.handlers.clear()
    # Recorre una copia superficial de la lista de handlers.
    # [:] porque vamos a quitar elementos de la lista original mientras iteramos.
    for handler in logger.handlers[:]:
        # Desvincula el handler del logger.
        logger.removeHandler(handler)
        # Cierra el handler para liberar recursos (archivos abiertos, etc.).
        handler.close()
        
    # No propagar al logger root de Jupyter para no ensuciar la salida estándar
    logger.propagate = False

    # 1. Handler para Consola (solo mensajes INFO o peores, formato simple)
    consola = logging.StreamHandler()
    consola.setLevel(logging.INFO)
    consola.setFormatter(logging.Formatter('%(levelname)s: %(message)s'))

    # 2. Handler para Archivo (registra TODO, modo escritura, formato completo)
    # El parámetro mode='w' abre el archivo en modo escritura y trunca su contenido anterior si ya existe.
    archivo_log = logging.FileHandler(archivo, mode='w', encoding='utf-8')
    archivo_log.setLevel(logging.DEBUG)
    archivo_log.setFormatter(logging.Formatter('%(asctime)s | %(levelname)s | %(message)s'))

    logger.addHandler(consola)
    logger.addHandler(archivo_log)

    return logger



In [ ]:

# #from pathlib import Path

# resultado = capturar_entorno(
#     Path("archivo_inexistente.csv"),
#     RAIZ / "requirements.txt"
# )

# print(resultado["hash_datos_csv"])
# """
# """
# resultado = capturar_entorno(
#     DATA_PATH,
#     RAIZ / "requirements.txt"
# )

# for clave, valor in resultado.items():
#     print(f"{clave}: {valor}")


In [ ]:

# for clave in ("hash_datos_csv", "hash_requirements"):
#     huella = resultado[clave]

#     assert len(huella) == 64, f"Longitud inválida: {clave}"
#     assert all(c in "0123456789abcdef" for c in huella), \
#         f"Formato hexadecimal inválido: {clave}"

#     print(f"{clave}: formato SHA-256 válido")


In [ ]:

# ## Test 1 
# from tempfile import TemporaryDirectory

# with TemporaryDirectory() as carpeta:
#     ruta_log = Path(carpeta) / "prueba_logger.log"

#     logger_prueba = crear_logger("test_logger_hito2", ruta_log)

#     handler_anterior = next(
#         h for h in logger_prueba.handlers
#         if isinstance(h, logging.FileHandler)
#     )

#     print("Handlers primera ejecución:", len(logger_prueba.handlers))

#     crear_logger("test_logger_hito2", ruta_log)

#     print("Handlers segunda ejecución:", len(logger_prueba.handlers))
#     print("Handler anterior cerrado:", handler_anterior.stream.closed)

#     # Limpieza de recursos utilizados por esta prueba.
#     for handler in logger_prueba.handlers[:]:
#         logger_prueba.removeHandler(handler)
#         handler.close()

#     handler_anterior.close()
# # Handlers primera ejecución: 2
# # Handlers segunda ejecución: 2
# # Handler anterior cerrado: False
# """
# ## Test 2
# ## despues de realizar los cambios y agregar el iterador 
# ## Vovler a ejecutar la celda de prueba y observar que el archivo asociado al FileHandler anterior queda cerrado. 
# # tenemos evidencia de que:
# #  Los handlers no se acumulan al volver a llamar a la función.
# ## Los handlers anteriores se desvinculan del logger.
# ## El archivo asociado al FileHandler anterior queda cerrado.
# ## La función puede volver a configurar sus handlers sin conservar abierto el archivo anterior en el caso probado.
# """
# from tempfile import TemporaryDirectory

# with TemporaryDirectory() as carpeta:
#     ruta_log = Path(carpeta) / "prueba_logger.log"

#     logger_prueba = crear_logger("test_logger_hito2", ruta_log)

#     handler_anterior = next(
#         h for h in logger_prueba.handlers
#         if isinstance(h, logging.FileHandler)
#     )

#     stream_anterior = handler_anterior.stream

#     print("Handlers primera ejecución:", len(logger_prueba.handlers))

#     crear_logger("test_logger_hito2", ruta_log)

#     print("Handlers segunda ejecución:", len(logger_prueba.handlers))
#     print("Archivo anterior cerrado:", stream_anterior.closed)

#     # Limpieza de recursos de la prueba
#     for handler in logger_prueba.handlers[:]:
#         logger_prueba.removeHandler(handler)
#         handler.close()
# ## La prueba confirma que solucionamos el problema de cierre de recursos.
# ##Todavía no demuestra que los mensajes se registren correctamente ni que conservemos los logs al realizar nuevas ejecuciones.


In [ ]:
# # continuar con la revisión de mode='w' y la persistencia de los logs
# # Me estan exijiendo
# # Con mode='w', el contenido anterior desaparece.
# # Con mode='a', los mensajes anteriores permanecen y los nuevos se agregan al final.
# # Si ejecutamos web_bimestre y después tienda_bimestre, cada corrida utilizará un archivo diferente.
# # Por lo tanto, mode='w' no supone un problema de sobrescritura entre corridas que utilizan distintos run_id.
# # El riesgo aparece cuando volvemos a ejecutar una corrida utilizando el mismo run_id.
# # La prueba es segura para nuestros artefactos porque utiliza un archivo temporal que será eliminado al finalizar

# from tempfile import TemporaryDirectory

# with TemporaryDirectory() as carpeta:
#     ruta = Path(carpeta) / "prueba.log"

#     logger = crear_logger("prueba_persistencia", ruta)
#     logger.info("Primera ejecución")

#     logger = crear_logger("prueba_persistencia", ruta)
#     logger.info("Segunda ejecución")

#     for handler in logger.handlers[:]:
#         logger.removeHandler(handler)
#         handler.close()

#     print("Contenido final del archivo:")
#     print(ruta.read_text(encoding="utf-8"))
  
# # INFO: Primera ejecución
# # INFO: Segunda ejecución
# # Contenido final del archivo:
# # 2026-10-18 00:46:08,020 | INFO | Segunda ejecución


In [ ]:
# ## Test de logger

# from tempfile import TemporaryDirectory
# from pathlib import Path
# import logging

# with TemporaryDirectory() as carpeta:
#     ruta_log = Path(carpeta) / "prueba_niveles.log"

#     logger_prueba = crear_logger("test_niveles_hito2", ruta_log)

#     # Generamos un mensaje por cada nivel
#     logger_prueba.debug("Mensaje de depuración")
#     logger_prueba.info("Inicio del procesamiento")
#     logger_prueba.warning("Se encontró un dato anómalo")
#     logger_prueba.error("Error durante el procesamiento")
#     logger_prueba.critical("Error crítico del sistema")

#     # Cerramos los handlers para finalizar la escritura
#     for handler in logger_prueba.handlers[:]:
#         logger_prueba.removeHandler(handler)
#         handler.close()

#     print("\n--- CONTENIDO DEL ARCHIVO ---")
#     contenido = ruta_log.read_text(encoding="utf-8")
#     print(contenido)

# #INFO: Inicio del procesamiento
# #WARNING: Se encontró un dato anómalo
# #ERROR: Error durante el procesamiento
# #CRITICAL: Error crítico del sistema

# #--- CONTENIDO DEL ARCHIVO ---
# #2026-10-18 00:57:39,907 | DEBUG | Mensaje de depuración
# #2026-10-18 00:57:39,907 | INFO | Inicio del procesamiento
# #2026-10-18 00:57:39,907 | WARNING | Se encontró un dato anómalo
# #2026-10-18 00:57:39,909 | ERROR | Error durante el procesamiento
# #2026-10-18 00:57:39,909 | CRITICAL | Error crítico del sistema

INFO: Inicio del procesamiento
ERROR: Error durante el procesamiento
CRITICAL: Error crítico del sistema



--- CONTENIDO DEL ARCHIVO ---
2026-10-18 00:57:39,907 | DEBUG | Mensaje de depuración
2026-10-18 00:57:39,907 | INFO | Inicio del procesamiento
2026-10-18 00:57:39,907 | WARNING | Se encontró un dato anómalo
2026-10-18 00:57:39,909 | ERROR | Error durante el procesamiento
2026-10-18 00:57:39,909 | CRITICAL | Error crítico del sistema



In [30]:
# TODO · Diseñá las comprobaciones del Hito 2.
# Tus pruebas deben detectar, como mínimo:
# - handlers duplicados al volver a crear el logger;
# - claves faltantes en el recibo de entorno;
# - una huella de datos que no tenga el formato esperado.
#
# Cada comprobación debe incluir un mensaje que permita corregir el problema.
# PRUEBA 1 - Detectar handlers duplicados

from tempfile import TemporaryDirectory

with TemporaryDirectory() as carpeta:
    ruta_log = Path(carpeta) / "prueba_handlers.log"

    logger_prueba = crear_logger("test_handlers_hito2", ruta_log)

    # Segunda creación utilizando el mismo nombre
    logger_prueba = crear_logger("test_handlers_hito2", ruta_log)

    try:
        assert len(logger_prueba.handlers) == 2, (
            "ERROR: Se detectaron handlers duplicados. "
            "Revisar la eliminación y cierre de handlers anteriores."
        )

        print("PRUEBA 1 OK: El logger mantiene exactamente 2 handlers.")

    finally:
        # Liberar los recursos utilizados en la prueba
        for handler in logger_prueba.handlers[:]:
            logger_prueba.removeHandler(handler)
            handler.close()

# PRUEBA 2 - Detectar claves faltantes en el recibo de entorno

recibo = capturar_entorno(DATA_PATH, RAIZ / "requirements.txt")

claves_obligatorias = {
    "sistema_operativo",
    "version_python",
    "version_pandas",
    "version_papermill",
    "hash_datos_csv",
    "hash_requirements"
}

faltantes = claves_obligatorias - set(recibo.keys())

assert not faltantes, (
    f"ERROR: Faltan claves en el recibo de entorno: {sorted(faltantes)}. "
    "Revisar el diccionario devuelto por capturar_entorno()."
)

print("PRUEBA 2 OK: El recibo contiene todas las claves obligatorias.")

# PRUEBA 3 - Validar formato de las huellas SHA-256

for clave in ("hash_datos_csv", "hash_requirements"):
    huella = recibo[clave]

    assert (
        isinstance(huella, str)
        and len(huella) == 64
        and all(c in "0123456789abcdef" for c in huella)
    ), (
        f"ERROR: La huella '{clave}' tiene un formato SHA-256 inválido. "
        "Verificar que sea una cadena de 64 caracteres hexadecimales "
        "y que se calcule mediante hashlib.sha256()."
    )

    print(f"PRUEBA 3 OK: {clave} tiene formato SHA-256 válido.")


# PRUEBA NEGATIVA - Simular una clave faltante

# Crear una copia del recibo original
recibo_incompleto = recibo.copy()

# Eliminar deliberadamente una clave obligatoria
recibo_incompleto.pop("version_papermill")

# Detectar las claves faltantes
faltantes_prueba = claves_obligatorias - set(recibo_incompleto.keys())

try:
    assert not faltantes_prueba, (
        f"ERROR: Faltan claves en el recibo: {sorted(faltantes_prueba)}. "
        "Revisar capturar_entorno()."
    )
except AssertionError as error:
    print("PRUEBA NEGATIVA OK: Se detectó el defecto.")
    print(error)
else:
    raise AssertionError("La prueba no detectó la clave faltante.")



# PRUEBA NEGATIVA - Detectar una huella SHA-256 inválida

# Copiamos el recibo original
recibo_invalido = recibo.copy()

# Introducimos deliberadamente una huella incorrecta
recibo_invalido["hash_datos_csv"] = "HASH_INVALIDO"

# Obtenemos la huella que queremos validar
huella = recibo_invalido["hash_datos_csv"]

try:
    assert (
        isinstance(huella, str)
        and len(huella) == 64
        and all(c in "0123456789abcdef" for c in huella)
    ), (
        "ERROR: La huella 'hash_datos_csv' tiene formato SHA-256 inválido. "
        "Verificar que sea una cadena de 64 caracteres hexadecimales "
        "y que se calcule mediante hashlib.sha256()."
    )

except AssertionError as error:
    print("PRUEBA NEGATIVA OK: Se detectó el hash inválido.")
    print(error)

else:
    raise AssertionError(
        "ERROR: La prueba no detectó la huella SHA-256 inválida."
    )

PRUEBA 1 OK: El logger mantiene exactamente 2 handlers.
PRUEBA 2 OK: El recibo contiene todas las claves obligatorias.
PRUEBA 3 OK: hash_datos_csv tiene formato SHA-256 válido.
PRUEBA 3 OK: hash_requirements tiene formato SHA-256 válido.
PRUEBA NEGATIVA OK: Se detectó el defecto.
ERROR: Faltan claves en el recibo: ['version_papermill']. Revisar capturar_entorno().
PRUEBA NEGATIVA OK: Se detectó el hash inválido.
ERROR: La huella 'hash_datos_csv' tiene formato SHA-256 inválido. Verificar que sea una cadena de 64 caracteres hexadecimales y que se calcule mediante hashlib.sha256().


## Hito 3 · Pipeline observable

Separá carga, limpieza, filtro y resumen. Cada decisión relevante debe quedar en el log.


In [38]:
# TODO 2 · Completá los cuerpos; conservá estas firmas.
def cargar_ventas(ruta: Path, logger: logging.Logger) -> pd.DataFrame:
    """Carga los datos y valida las columnas requeridas."""
    df = pd.read_csv(ruta)
    # columnas_esperadas = {"fecha", "canal", "producto", "unidades", "precio_unitario"}
    columnas_esperadas = {"fecha", "canal", "categoria", "unidades", "precio_unitario"}
    columnas_actuales = set(df.columns)
    
    if not columnas_esperadas.issubset(columnas_actuales):
        logger.error(f"Columnas faltantes. Esperadas: {columnas_esperadas}, Actuales: {columnas_actuales}")
        raise ValueError("El archivo no contiene las cinco columnas requeridas.")
        
    logger.info(f"Datos cargados correctamente: {len(df)} registros encontrados.")
    return df

def limpiar_ventas(df: pd.DataFrame, logger: logging.Logger) -> pd.DataFrame:
    """Aplica reglas de calidad, registrando descartes e imputaciones."""
    df_limpio = df.copy()
    
    # 1. Normalizar `Wb` como `Web`
    mask_wb = df_limpio['canal'] == 'Wb'
    if mask_wb.sum() > 0:
        df_limpio.loc[mask_wb, 'canal'] = 'Web'
        logger.warning(f"Se normalizaron {mask_wb.sum()} registros de 'Wb' a 'Web'.")

    # 2. Convertir fechas y descartar inválidas
    df_limpio['fecha'] = pd.to_datetime(df_limpio['fecha'], errors="coerce")
    fechas_nulas = df_limpio['fecha'].isna().sum()
    if fechas_nulas > 0:
        df_limpio = df_limpio.dropna(subset=['fecha'])
        logger.info(f"Se descartaron {fechas_nulas} registros por fechas inválidas.")

    # 3. Descartar unidades <= 0
    mask_unidades = df_limpio['unidades'] <= 0
    if mask_unidades.sum() > 0:
        df_limpio = df_limpio[~mask_unidades]
        logger.info(f"Se descartaron {mask_unidades.sum()} registros con unidades <= 0.")

    # 4. Imputar precios faltantes con la mediana
    precios_nulos = df_limpio['precio_unitario'].isna().sum()
    if precios_nulos > 0:
        mediana = df_limpio['precio_unitario'].median()
        if pd.isna(mediana):
            logger.error("No se puede imputar precios: la mediana es NaN (no hay datos válidos).")
            raise ValueError("La mediana de precios es NaN.")
        df_limpio['precio_unitario'] = df_limpio['precio_unitario'].fillna(mediana)
        logger.info(f"Se imputaron {precios_nulos} precios faltantes usando la mediana del set ({mediana}).")

    # 5. Crear columna venta_total
    df_limpio['venta_total'] = df_limpio['unidades'] * df_limpio['precio_unitario']
    
    return df_limpio

def filtrar_ventas(
    df: pd.DataFrame, canal_elegido: str, inicio: str, fin: str, logger: logging.Logger
) -> pd.DataFrame:
    """Filtra los datos y detiene la corrida si el resultado es vacío."""
    df_filtrado = df.copy()
    
    inicio_dt = pd.to_datetime(inicio)
    fin_dt = pd.to_datetime(fin)
    
    mask = (df_filtrado['canal'] == canal_elegido) & (df_filtrado['fecha'] >= inicio_dt) & (df_filtrado['fecha'] <= fin_dt)
    df_filtrado = df_filtrado[mask]
    
    if df_filtrado.empty:
        logger.error(f"Filtro vacío para canal '{canal_elegido}' entre {inicio} y {fin}.")
        raise ValueError("El resultado del filtro está vacío.")
        
    logger.info(f"Filtro aplicado exitosamente. Quedan {len(df_filtrado)} registros.")
    return df_filtrado

def resumir_ventas(df: pd.DataFrame, parametros: dict) -> dict:
    """Genera el diccionario de resumen con totales."""
    return {
        "parametros": parametros,
        "registros_finales": len(df),
        "venta_total": float(df['venta_total'].sum())
    }

In [ ]:
# ## Primera prueba: detectar una columna faltante
# from tempfile import TemporaryDirectory

# with TemporaryDirectory() as carpeta:
#     ruta = Path(carpeta)

#     # Dataset deliberadamente incompleto
#     df_prueba = pd.DataFrame({
#         "fecha": ["2026-01-15"],
#         "canal": ["Web"],
#         "unidades": [2],
#         "precio_unitario": [150.0]
#     })

#     archivo_csv = ruta / "ventas_incompletas.csv"
#     df_prueba.to_csv(archivo_csv, index=False)

#     logger_prueba = crear_logger(
#         "test_carga_hito3",
#         ruta / "prueba_carga.log"
#     )

#     try:
#         cargar_ventas(archivo_csv, logger_prueba)

#     except ValueError as error:
#         print("PRUEBA NEGATIVA OK: Se detectó el esquema incompleto.")
#         print(error)

#     else:
#         raise AssertionError(
#             "ERROR: cargar_ventas() aceptó un CSV sin la columna categoria."
#         )

#     finally:
#         for handler in logger_prueba.handlers[:]:
#             logger_prueba.removeHandler(handler)
#             handler.close()            

ERROR: Columnas faltantes. Esperadas: {'unidades', 'fecha', 'precio_unitario', 'canal', 'categoria'}, Actuales: {'canal', 'unidades', 'fecha', 'precio_unitario'}


PRUEBA NEGATIVA OK: Se detectó el esquema incompleto.
El archivo no contiene las cinco columnas requeridas.


In [ ]:
# # HITO 3 - PRUEBA POSITIVA DE cargar_ventas()

# from tempfile import TemporaryDirectory

# with TemporaryDirectory() as carpeta:
#     ruta_log = Path(carpeta) / "prueba_carga.log"
#     logger_prueba = crear_logger("test_carga_positiva", ruta_log)

#     try:
#         # Ejecutar la función con el dataset real
#         df_prueba = cargar_ventas(DATA_PATH, logger_prueba)

#         # PRUEBA 1: Tipo de resultado
#         assert isinstance(df_prueba, pd.DataFrame), (
#             "ERROR: cargar_ventas debe devolver un DataFrame."
#         )

#         # PRUEBA 2: Cantidad de registros
#         assert len(df_prueba) == 120, (
#             f"ERROR: Se esperaban 120 registros y se obtuvieron {len(df_prueba)}."
#         )

#         # PRUEBA 3: Columnas obligatorias
#         columnas_obligatorias = {
#             "fecha", "canal", "categoria",
#             "unidades", "precio_unitario"
#         }

#         faltantes = columnas_obligatorias - set(df_prueba.columns)

#         assert not faltantes, (
#             f"ERROR: Faltan columnas obligatorias: {sorted(faltantes)}"
#         )

#     finally:
#         # Liberar los handlers
#         for handler in logger_prueba.handlers[:]:
#             logger_prueba.removeHandler(handler)
#             handler.close()

#     # PRUEBA 4: Verificar que el evento quedó registrado
#     contenido_log = ruta_log.read_text(encoding="utf-8")

#     assert "120 registros encontrados" in contenido_log, (
#         "ERROR: No se registró correctamente la cantidad de filas."
#     )

#     print("PRUEBA 1 OK: Se devolvió un DataFrame.")
#     print("PRUEBA 2 OK: Se cargaron 120 registros.")
#     print("PRUEBA 3 OK: Las cinco columnas obligatorias están presentes.")
#     print("PRUEBA 4 OK: El logger registró la cantidad de filas.")
#     print("\nColumnas:", df_prueba.columns.tolist())
#     print("Dimensiones:", df_prueba.shape)



INFO: Datos cargados correctamente: 120 registros encontrados.


PRUEBA 1 OK: Se devolvió un DataFrame.
PRUEBA 2 OK: Se cargaron 120 registros.
PRUEBA 3 OK: Las cinco columnas obligatorias están presentes.
PRUEBA 4 OK: El logger registró la cantidad de filas.

Columnas: ['fecha', 'canal', 'categoria', 'unidades', 'precio_unitario']
Dimensiones: (120, 5)


In [ ]:
# # HITO 3 - PRUEBAS DE limpiar_ventas()

# from tempfile import TemporaryDirectory

# with TemporaryDirectory() as carpeta:
#     ruta_log = Path(carpeta) / "prueba_limpieza.log"
#     logger_prueba = crear_logger("test_limpieza_hito3", ruta_log)

#     try:
#         # Datos originales y copia para verificar integridad
#         df_original = pd.read_csv(DATA_PATH)
#         df_respaldo = df_original.copy(deep=True)

#         # Ejecutar nuestra función
#         df_limpio = limpiar_ventas(df_original, logger_prueba)

#         # PRUEBA 1 - No modificar el DataFrame original
#         pd.testing.assert_frame_equal(df_original, df_respaldo)
#         print("PRUEBA 1 OK: DataFrame original conservado.")

#         # PRUEBA 2 - Normalización del canal Wb
#         indices_wb = df_original.index[df_original["canal"] == "Wb"]

#         assert df_limpio.loc[indices_wb, "canal"].eq("Web").all(), (
#             "ERROR: No se normalizaron correctamente los canales Wb."
#         )
#         assert not df_limpio["canal"].eq("Wb").any()
#         print("PRUEBA 2 OK: Canal Wb normalizado a Web.")

#         # PRUEBA 3 - Fechas válidas
#         assert pd.api.types.is_datetime64_any_dtype(df_limpio["fecha"]), (
#             "ERROR: La columna fecha no tiene tipo datetime."
#         )
#         assert df_limpio["fecha"].notna().all(), (
#             "ERROR: Persisten fechas inválidas."
#         )
#         print("PRUEBA 3 OK: Fechas convertidas y válidas.")

#         # PRUEBA 4 - Unidades positivas
#         assert (df_limpio["unidades"] > 0).all(), (
#             "ERROR: Persisten registros con unidades <= 0."
#         )
#         print("PRUEBA 4 OK: Todas las unidades son positivas.")

#         # PRUEBA 5 - Imputación de precios
#         assert df_limpio["precio_unitario"].notna().all(), (
#             "ERROR: Persisten precios faltantes."
#         )

#         indices_nulos = df_original.index[
#             df_original["precio_unitario"].isna()
#         ]

#         assert df_limpio.loc[
#             indices_nulos, "precio_unitario"
#         ].eq(7500.0).all(), (
#             "ERROR: Los precios faltantes no se imputaron "
#             "con la mediana esperada."
#         )
#         print("PRUEBA 5 OK: Precios imputados con mediana 7500.0.")

#         # PRUEBA 6 - Cálculo de venta_total
#         esperado = (
#             df_limpio["unidades"] * df_limpio["precio_unitario"]
#         )

#         assert "venta_total" in df_limpio.columns, (
#             "ERROR: Falta la columna venta_total."
#         )

#         pd.testing.assert_series_equal(
#             df_limpio["venta_total"],
#             esperado.rename("venta_total")
#         )
#         print("PRUEBA 6 OK: venta_total calculado correctamente.")

#         # PRUEBA 7 - Cantidad de registros
#         assert len(df_limpio) == 116, (
#             f"ERROR: Se esperaban 116 filas y se obtuvieron "
#             f"{len(df_limpio)}."
#         )
#         print("PRUEBA 7 OK: 116 registros finales.")

#         # PRUEBA 8 - Caso negativo: mediana NaN
#         df_sin_precios = pd.DataFrame({
#             "fecha": ["2026-01-01", "2026-01-02"],
#             "canal": ["Web", "Tienda"],
#             "categoria": ["Hogar", "Ropa"],
#             "unidades": [2, 3],
#             "precio_unitario": [float("nan"), float("nan")]
#         })

#         error_detectado = False

#         try:
#             limpiar_ventas(df_sin_precios, logger_prueba)
#         except ValueError as error:
#             assert "mediana" in str(error).lower(), (
#                 "ERROR: ValueError no explica el problema de la mediana."
#             )
#             error_detectado = True

#         assert error_detectado, (
#             "ERROR: No se generó ValueError cuando la mediana es NaN."
#         )
#         print("PRUEBA 8 OK: Mediana NaN genera ValueError.")

#     finally:
#         for handler in logger_prueba.handlers[:]:
#             logger_prueba.removeHandler(handler)
#             handler.close()

#     # PRUEBA 9 - Verificación del log
#     contenido_log = ruta_log.read_text(encoding="utf-8")

#     mensajes_esperados = [
#         ("WARNING", "Se normalizaron 1 registros"),
#         ("INFO", "Se descartaron 1 registros por fechas inválidas"),
#         ("INFO", "Se descartaron 3 registros con unidades <= 0"),
#         ("INFO", "Se imputaron 4 precios faltantes"),
#         ("ERROR", "No se puede imputar precios")
#     ]

#     for nivel, mensaje in mensajes_esperados:
#         assert f"| {nivel} | {mensaje}" in contenido_log, (
#             f"ERROR: No se encontró en el log: {nivel} - {mensaje}"
#         )

#     print("PRUEBA 9 OK: Eventos de calidad registrados en el log.")

#     print("\nRESUMEN:")
#     print("Registros originales:", len(df_original))
#     print("Registros finales:", len(df_limpio))
#     print("Registros descartados:", len(df_original) - len(df_limpio))
#     print("Columnas finales:", df_limpio.columns.tolist())

INFO: Se descartaron 1 registros por fechas inválidas.
INFO: Se descartaron 3 registros con unidades <= 0.
INFO: Se imputaron 4 precios faltantes usando la mediana del set (7500.0).
c:\codigo\python\ambientes\ProgramacionAvanzada_Reto-Modulo-4\.venv\Lib\site-packages\numpy\lib\_nanfunctions_impl.py:1213: RuntimeWarning: Mean of empty slice
  return np.nanmean(a, axis, out=out, keepdims=keepdims)
ERROR: No se puede imputar precios: la mediana es NaN (no hay datos válidos).


PRUEBA 1 OK: DataFrame original conservado.
PRUEBA 2 OK: Canal Wb normalizado a Web.
PRUEBA 3 OK: Fechas convertidas y válidas.
PRUEBA 4 OK: Todas las unidades son positivas.
PRUEBA 5 OK: Precios imputados con mediana 7500.0.
PRUEBA 6 OK: venta_total calculado correctamente.
PRUEBA 7 OK: 116 registros finales.
PRUEBA 8 OK: Mediana NaN genera ValueError.
PRUEBA 9 OK: Eventos de calidad registrados en el log.

RESUMEN:
Registros originales: 120
Registros finales: 116
Registros descartados: 4
Columnas finales: ['fecha', 'canal', 'categoria', 'unidades', 'precio_unitario', 'venta_total']


In [ ]:
# # HITO 3 - PRUEBAS DE filtrar_ventas()

# from tempfile import TemporaryDirectory

# with TemporaryDirectory() as carpeta:
#     ruta_log = Path(carpeta) / "prueba_filtros.log"
#     logger_prueba = crear_logger("test_filtros_hito3", ruta_log)

#     try:
#         # Preparar datos reales
#         df_original = cargar_ventas(DATA_PATH, logger_prueba)
#         df_limpio = limpiar_ventas(df_original, logger_prueba)
#         df_respaldo = df_limpio.copy(deep=True)

#         # PRUEBA 1 - Canal Web, primer bimestre
#         df_web = filtrar_ventas(
#             df_limpio, "Web",
#             "2026-01-01", "2026-02-28", logger_prueba
#         )

#         assert len(df_web) == 13, (
#             f"ERROR: Se esperaban 13 registros Web, "
#             f"se obtuvieron {len(df_web)}."
#         )
#         print("PRUEBA 1 OK: Canal Web, 13 registros.")

#         # PRUEBA 2 - Canal Tienda, primer bimestre
#         df_tienda = filtrar_ventas(
#             df_limpio, "Tienda",
#             "2026-01-01", "2026-02-28", logger_prueba
#         )

#         assert len(df_tienda) == 19, (
#             f"ERROR: Se esperaban 19 registros Tienda, "
#             f"se obtuvieron {len(df_tienda)}."
#         )
#         print("PRUEBA 2 OK: Canal Tienda, 19 registros.")

#         # PRUEBA 3 - Condiciones de canal y fechas
#         assert df_web["canal"].eq("Web").all(), (
#             "ERROR: El filtro Web contiene otros canales."
#         )

#         assert df_web["fecha"].between(
#             "2026-01-01", "2026-02-28"
#         ).all(), (
#             "ERROR: Hay fechas fuera del período."
#         )

#         print("PRUEBA 3 OK: Canal y período correctos.")

#         # PRUEBA 4 - Límites inclusivos
#         df_limites = pd.DataFrame({
#             "fecha": pd.to_datetime([
#                 "2026-01-01",
#                 "2026-02-28",
#                 "2026-03-01",
#                 "2026-02-01"
#             ]),
#             "canal": ["Web", "Web", "Web", "Tienda"]
#         })

#         df_limites_filtrado = filtrar_ventas(
#             df_limites, "Web",
#             "2026-01-01", "2026-02-28", logger_prueba
#         )

#         assert len(df_limites_filtrado) == 2, (
#             "ERROR: El filtro no respeta los límites inclusivos."
#         )

#         assert set(df_limites_filtrado["fecha"]) == {
#             pd.Timestamp("2026-01-01"),
#             pd.Timestamp("2026-02-28")
#         }, "ERROR: No se conservaron ambas fechas límite."

#         print("PRUEBA 4 OK: Fechas inicial y final incluidas.")

#         # PRUEBA 5 - Integridad del DataFrame recibido
#         pd.testing.assert_frame_equal(df_limpio, df_respaldo)

#         print("PRUEBA 5 OK: DataFrame original conservado.")

#         # PRUEBA 6 - Filtro sin registros
#         error_detectado = False

#         try:
#             filtrar_ventas(
#                 df_limpio, "CanalInexistente",
#                 "2026-01-01", "2026-02-28", logger_prueba
#             )

#         except ValueError as error:
#             assert "vacío" in str(error).lower(), (
#                 "ERROR: La excepción no explica el filtro vacío."
#             )
#             error_detectado = True

#         assert error_detectado, (
#             "ERROR: El filtro vacío no generó ValueError."
#         )

#         print("PRUEBA 6 OK: Filtro vacío detectado.")

#     finally:
#         for handler in logger_prueba.handlers[:]:
#             logger_prueba.removeHandler(handler)
#             handler.close()

#     # PRUEBA 7 - Observabilidad
#     contenido_log = ruta_log.read_text(encoding="utf-8")

#     assert "| ERROR | Filtro vacío" in contenido_log, (
#         "ERROR: No se registró el filtro vacío en el log."
#     )

#     assert "Quedan 13 registros" in contenido_log, (
#         "ERROR: No se registró el resultado del filtro Web."
#     )

#     assert "Quedan 19 registros" in contenido_log, (
#         "ERROR: No se registró el resultado del filtro Tienda."
#     )

#     print("PRUEBA 7 OK: Eventos registrados en el log.")

#     print("\nRESUMEN:")
#     print("Registros limpios:", len(df_limpio))
#     print("Registros Web:", len(df_web))
#     print("Registros Tienda:", len(df_tienda))
    

INFO: Datos cargados correctamente: 120 registros encontrados.
INFO: Se descartaron 1 registros por fechas inválidas.
INFO: Se descartaron 3 registros con unidades <= 0.
INFO: Se imputaron 4 precios faltantes usando la mediana del set (7500.0).
INFO: Filtro aplicado exitosamente. Quedan 13 registros.
INFO: Filtro aplicado exitosamente. Quedan 19 registros.
INFO: Filtro aplicado exitosamente. Quedan 2 registros.
ERROR: Filtro vacío para canal 'CanalInexistente' entre 2026-01-01 y 2026-02-28.


PRUEBA 1 OK: Canal Web, 13 registros.
PRUEBA 2 OK: Canal Tienda, 19 registros.
PRUEBA 3 OK: Canal y período correctos.
PRUEBA 4 OK: Fechas inicial y final incluidas.
PRUEBA 5 OK: DataFrame original conservado.
PRUEBA 6 OK: Filtro vacío detectado.
PRUEBA 7 OK: Eventos registrados en el log.

RESUMEN:
Registros limpios: 116
Registros Web: 13
Registros Tienda: 19


In [ ]:
# # HITO 3 - PRUEBAS DE resumir_ventas()

# from tempfile import TemporaryDirectory
# from math import isclose

# with TemporaryDirectory() as carpeta:
#     ruta_log = Path(carpeta) / "prueba_resumen.log"
#     logger_prueba = crear_logger("test_resumen_hito3", ruta_log)

#     try:
#         # Preparar los datos reales
#         df_original = cargar_ventas(DATA_PATH, logger_prueba)
#         df_limpio = limpiar_ventas(df_original, logger_prueba)

#         # Probar dos configuraciones
#         casos = [
#             ("Web", 13, 382500.0),
#             ("Tienda", 19, 797500.0)
#         ]

#         for canal_prueba, filas_esperadas, total_esperado in casos:

#             parametros_prueba = {
#                 "canal": canal_prueba,
#                 "fecha_inicio": "2026-01-01",
#                 "fecha_fin": "2026-02-28",
#                 "run_id": f"prueba_{canal_prueba.lower()}"
#             }

#             df_filtrado = filtrar_ventas(
#                 df_limpio,
#                 canal_prueba,
#                 "2026-01-01",
#                 "2026-02-28",
#                 logger_prueba
#             )

#             respaldo = df_filtrado.copy(deep=True)

#             # Ejecutar la función que estamos auditando
#             resumen = resumir_ventas(df_filtrado, parametros_prueba)

#             # PRUEBA 1 - Tipo y estructura
#             assert isinstance(resumen, dict), (
#                 "ERROR: resumir_ventas debe devolver un diccionario."
#             )

#             claves = {"parametros", "registros_finales", "venta_total"}

#             assert claves.issubset(resumen), (
#                 f"ERROR: Faltan claves: {sorted(claves - set(resumen))}"
#             )

#             # PRUEBA 2 - Parámetros
#             assert resumen["parametros"] == parametros_prueba, (
#                 "ERROR: Los parámetros del resumen no coinciden."
#             )

#             # PRUEBA 3 - Cantidad de registros
#             assert resumen["registros_finales"] == filas_esperadas, (
#                 f"ERROR: Cantidad incorrecta para {canal_prueba}."
#             )

#             # PRUEBA 4 - Venta total
#             assert isclose(
#                 resumen["venta_total"],
#                 total_esperado,
#                 rel_tol=0,
#                 abs_tol=0.01
#             ), (
#                 f"ERROR: Venta total incorrecta para {canal_prueba}. "
#                 f"Esperado: {total_esperado}, "
#                 f"Obtenido: {resumen['venta_total']}"
#             )

#             # PRUEBA 5 - Integridad del DataFrame
#             pd.testing.assert_frame_equal(df_filtrado, respaldo)

#             # PRUEBA 6 - Compatibilidad con JSON
#             json.dumps(resumen, ensure_ascii=False)

#             print(f"\n{canal_prueba}: PRUEBAS 1-6 OK")
#             print("Registros:", resumen["registros_finales"])
#             print("Venta total:", resumen["venta_total"])
#             print("Parámetros:", resumen["parametros"])

#     finally:
#         for handler in logger_prueba.handlers[:]:
#             logger_prueba.removeHandler(handler)
#             handler.close()


INFO: Datos cargados correctamente: 120 registros encontrados.
INFO: Se descartaron 1 registros por fechas inválidas.
INFO: Se descartaron 3 registros con unidades <= 0.
INFO: Se imputaron 4 precios faltantes usando la mediana del set (7500.0).
INFO: Filtro aplicado exitosamente. Quedan 13 registros.
INFO: Filtro aplicado exitosamente. Quedan 19 registros.



Web: PRUEBAS 1-6 OK
Registros: 13
Venta total: 382500.0
Parámetros: {'canal': 'Web', 'fecha_inicio': '2026-01-01', 'fecha_fin': '2026-02-28', 'run_id': 'prueba_web'}

Tienda: PRUEBAS 1-6 OK
Registros: 19
Venta total: 797500.0
Parámetros: {'canal': 'Tienda', 'fecha_inicio': '2026-01-01', 'fecha_fin': '2026-02-28', 'run_id': 'prueba_tienda'}


In [41]:
# TODO · Diseñá las comprobaciones del Hito 3.
# Construí una muestra pequeña y verificá, como mínimo:
# - descarte de fechas y unidades inválidas;
# - normalización de Wb como Web;
# - imputación de precios faltantes;
# - filtro por canal y rango;
# - coherencia del resumen calculado.
# A diferencia de las pruebas anteriores, aquí conocemos exactamente cada registro de entrada y su resultado esperado.
# Por ejemplo, sabemos que la fila 1 tiene tres unidades y un precio faltante. Una vez imputado 
# con la mediana de 300, su venta total debe ser 900.
# No estamos confiando únicamente en que pandas produzca un número: estamos comparando el resultado contra 
# un valor que calculamos de antemano.
# Además, la celda genera mensajes que ayudan a localizar cada posible problema, tal como buscamos 
# en las pruebas automáticas.

# HITO 3 - COMPROBACIONES AUTOMÁTICAS

from tempfile import TemporaryDirectory

with TemporaryDirectory() as carpeta:
    logger_test = crear_logger(
        "test_hito3",
        Path(carpeta) / "pruebas_hito3.log"
    )

    try:
        # Muestra controlada
        muestra = pd.DataFrame({
            "fecha": [
                "2026-01-01", "2026-01-15",
                "fecha-invalida", "2026-01-20",
                "2026-02-28", "2026-03-01",
                "2026-02-10"
            ],
            "canal": [
                "Wb", "Web", "Web", "Tienda",
                "Web", "Web", "Tienda"
            ],
            "categoria": ["A", "B", "C", "D", "E", "F", "G"],
            "unidades": [2, 3, 5, 0, 1, 4, 2],
            "precio_unitario": [
                100.0, None, 300.0, 900.0,
                200.0, 400.0, 600.0
            ]
        })

        # Ejecutar limpieza
        limpio = limpiar_ventas(muestra, logger_test)

        # PRUEBA 1 - Descartes
        assert len(limpio) == 5, (
            "ERROR: La limpieza debe conservar 5 registros. "
            "Revisar descartes por fecha y unidades."
        )
        assert 2 not in limpio.index and 3 not in limpio.index, (
            "ERROR: No se descartaron correctamente "
            "la fecha inválida o las unidades no positivas."
        )
        print("PRUEBA 1 OK: Descartes correctos.")

        # PRUEBA 2 - Normalización
        assert limpio.loc[0, "canal"] == "Web", (
            "ERROR: No se normalizó Wb como Web."
        )
        print("PRUEBA 2 OK: Normalización de canal.")

        # PRUEBA 3 - Imputación por mediana
        assert limpio.loc[1, "precio_unitario"] == 300.0, (
            "ERROR: Precio faltante incorrectamente imputado. "
            "La mediana esperada es 300."
        )
        assert limpio["precio_unitario"].notna().all(), (
            "ERROR: Persisten precios faltantes."
        )
        print("PRUEBA 3 OK: Imputación por mediana.")

        # PRUEBA 4 - Cálculo de venta_total
        assert limpio.loc[1, "venta_total"] == 900.0, (
            "ERROR: venta_total no coincide con "
            "unidades * precio_unitario."
        )
        print("PRUEBA 4 OK: Cálculo de venta_total.")

        # Ejecutar filtro
        filtrado = filtrar_ventas(
            limpio, "Web",
            "2026-01-01", "2026-02-28",
            logger_test
        )

        # PRUEBA 5 - Canal y período
        assert set(filtrado.index) == {0, 1, 4}, (
            "ERROR: El filtro no devolvió los registros "
            "esperados para Web en el primer bimestre."
        )
        assert filtrado["canal"].eq("Web").all(), (
            "ERROR: El filtro conserva canales incorrectos."
        )
        print("PRUEBA 5 OK: Filtro por canal y fechas.")

        # Ejecutar resumen
        parametros_test = {
            "canal": "Web",
            "fecha_inicio": "2026-01-01",
            "fecha_fin": "2026-02-28"
        }

        resumen_test = resumir_ventas(
            filtrado, parametros_test
        )

        # PRUEBA 6 - Coherencia del resumen
        assert resumen_test["registros_finales"] == 3, (
            "ERROR: El resumen debe contener 3 registros."
        )
        assert resumen_test["venta_total"] == 1300.0, (
            "ERROR: La venta total esperada es 1300."
        )
        assert resumen_test["parametros"] == parametros_test, (
            "ERROR: El resumen no conserva los parámetros."
        )
        print("PRUEBA 6 OK: Resumen coherente.")

    finally:
        for handler in logger_test.handlers[:]:
            logger_test.removeHandler(handler)
            handler.close()

    # PRUEBA 7 - Observabilidad
    log = (Path(carpeta) / "pruebas_hito3.log").read_text(
        encoding="utf-8"
    )

    assert "WARNING" in log and "Wb" in log, (
        "ERROR: No se registró la normalización de Wb."
    )
    assert "Se descartaron 1 registros por fechas inválidas" in log, (
        "ERROR: Falta el descarte de fechas en el log."
    )
    assert "Se descartaron 1 registros con unidades <= 0" in log, (
        "ERROR: Falta el descarte de unidades en el log."
    )
    assert "Se imputaron 1 precios faltantes" in log, (
        "ERROR: Falta registrar la imputación de precios."
    )

    print("PRUEBA 7 OK: Logging de calidad verificado.")

print("\nHITO 3: Todas las comprobaciones finalizaron correctamente.")

INFO: Se descartaron 1 registros por fechas inválidas.
INFO: Se descartaron 1 registros con unidades <= 0.
INFO: Se imputaron 1 precios faltantes usando la mediana del set (300.0).
INFO: Filtro aplicado exitosamente. Quedan 3 registros.


PRUEBA 1 OK: Descartes correctos.
PRUEBA 2 OK: Normalización de canal.
PRUEBA 3 OK: Imputación por mediana.
PRUEBA 4 OK: Cálculo de venta_total.
PRUEBA 5 OK: Filtro por canal y fechas.
PRUEBA 6 OK: Resumen coherente.
PRUEBA 7 OK: Logging de calidad verificado.

HITO 3: Todas las comprobaciones finalizaron correctamente.


### Orquestación y artefactos

Una corrida produce una carpeta propia. El notebook no debe depender de variables creadas fuera de esta secuencia.


In [52]:
# TODO 3 · Completá la orquestación dentro del try.
# Orquestá una corrida completa:
# 1. Creá RESULTADOS_DIR / run_id.
# 2. Inicializá el logger dentro de esa carpeta.
# 3. Registrá los parámetros sin datos personales.
# 4. Cargá, limpiá, filtrá y resumí.
# 5. Guardá detalle.csv, resumen.json y entorno.json.
# 6. Conservá en variables `detalle`, `resumen`, `entorno` y `RUN_DIR`.

#PARAMETROS = {"canal": canal, "fecha_inicio": fecha_inicio, "fecha_fin": fecha_fin, "run_id": run_id}
#RESULTADOS_DIR = RAIZ / directorio_salidas
#RUN_DIR = RESULTADOS_DIR / run_id
#RUN_DIR.mkdir(parents=True, exist_ok=True)
#logger = crear_logger(f"reto_integrador.{run_id}", RUN_DIR / "ejecucion.log")

#try:
    ####################################################################################################################
    # test  3 · Completá la orquestación dentro del try.
    ####################################################################################################################
# TODO 3 - Orquestación completa del pipeline
#
# Objetivo:
# Ejecutar el procesamiento completo y generar los cuatro artefactos
# asociados a un identificador único de corrida (run_id).


# ============================================================
# 1. CONFIGURACIÓN DE PARÁMETROS
# ============================================================

# Recuperamos los parámetros definidos en la celda "parameters".
# Papermill puede sobrescribirlos durante una ejecución parametrizada.

PARAMETROS = {
    "canal": canal,
    "fecha_inicio": fecha_inicio,
    "fecha_fin": fecha_fin,
    "run_id": run_id
    
}


# ============================================================
# 2. PREPARACIÓN DE DIRECTORIOS
# ============================================================

# Directorio general donde se almacenarán las ejecuciones.
RESULTADOS_DIR = RAIZ / directorio_salidas

# Cada corrida tendrá su propio directorio identificado por run_id.
RUN_DIR = RESULTADOS_DIR / run_id

# Crear el directorio si no existe.
# parents=True: crea también los directorios intermedios.
# exist_ok=True: no genera error si el directorio ya existe.
# El de abajo crear la carpeta que se pisa lo comento
# RUN_DIR.mkdir(parents=True, exist_ok=True)

# ============================================================
# Agrego la protección contra sobrescrituras
try:
    RUN_DIR.mkdir(parents=True, exist_ok=False)
except FileExistsError:
    raise FileExistsError(
        f"ERROR: El run_id '{run_id}' ya existe en {RUN_DIR}. "
        "Utilizá un run_id nuevo para conservar las evidencias anteriores."
    )

# ============================================================
# 3. INICIALIZACIÓN DEL LOGGER
# ============================================================

# Creamos el logger una sola vez.
# INFO y superiores se mostrarán en consola.
# DEBUG y superiores se registrarán en ejecucion.log.

logger = crear_logger(
    f"reto_integrador.{run_id}",
    RUN_DIR / "ejecucion.log"
)


# ============================================================
# 4. EJECUCIÓN DEL PIPELINE
# ============================================================

try:

    # Registrar el comienzo de la corrida y sus parámetros.
    logger.info(
        f"Iniciando corrida {run_id} con parámetros: {PARAMETROS}"
    )

    # --------------------------------------------------------
    # 4.1 CARGA DE DATOS
    # --------------------------------------------------------

    # Leer el CSV y validar las columnas obligatorias.
    df_crudo = cargar_ventas(DATA_PATH, logger)


    # --------------------------------------------------------
    # 4.2 LIMPIEZA DE DATOS
    # --------------------------------------------------------

    # Aplicar las reglas de calidad:
    # - Normalizar Wb a Web.
    # - Descartar fechas inválidas.
    # - Descartar unidades <= 0.
    # - Imputar precios faltantes con la mediana.
    # - Calcular venta_total.

    df_limpio = limpiar_ventas(df_crudo, logger)


    # --------------------------------------------------------
    # 4.3 FILTRADO DE DATOS
    # --------------------------------------------------------

    # Aplicar los parámetros de canal y período.
    # Si el resultado está vacío, la función genera ValueError.

    detalle = filtrar_ventas(
        df_limpio,
        canal,
        fecha_inicio,
        fecha_fin,
        logger
    )


    # --------------------------------------------------------
    # 4.4 RESUMEN DE VENTAS
    # --------------------------------------------------------

    # Generar el diccionario con los parámetros,
    # la cantidad de registros y el total de ventas.

    resumen = resumir_ventas(detalle, PARAMETROS)


    # --------------------------------------------------------
    # 4.5 CAPTURA DEL ENTORNO
    # --------------------------------------------------------

    # Identificar requirements.txt mediante una ruta relativa.
    req_path = RAIZ / "requirements.txt"

    # Capturar versiones del entorno y hashes SHA-256.
    entorno = capturar_entorno(DATA_PATH, req_path)


    # ========================================================
    # 5. GENERACIÓN DE ARTEFACTOS
    # ========================================================

    # ARTEFACTO 1: detalle.csv
    # Conservar los registros resultantes del procesamiento.

    detalle.to_csv(
        RUN_DIR / "detalle.csv",
        index=False
    )

    # ARTEFACTO 2: resumen.json
    # Conservar los parámetros y resultados agregados.

    with open(
        RUN_DIR / "resumen.json", "w", encoding="utf-8"
    ) as f:
        json.dump(resumen, f, ensure_ascii=False, indent=2)

    # ARTEFACTO 3: entorno.json
    # Conservar versiones del entorno y hashes de entrada.

    with open(
        RUN_DIR / "entorno.json", "w", encoding="utf-8"
    ) as f:
        json.dump(entorno, f, ensure_ascii=False, indent=2)

    # ARTEFACTO 4: ejecucion.log
    # Ya fue creado mediante crear_logger().
    # Registrar la finalización satisfactoria del procesamiento.

    logger.info(
        "Pipeline finalizado exitosamente. Artefactos guardados."
    )


# ============================================================
# 6. MANEJO DE ERRORES
# ============================================================

except Exception:

    # Registrar el error y su traceback completo.
    logger.exception(
        "La corrida falló. run_id=%s", run_id
    )

    # Volver a lanzar la excepción para que Jupyter/Papermill
    # identifique la ejecución como fallida.
    raise

INFO: Iniciando corrida manual_web_revision_01 con parámetros: {'canal': 'Web', 'fecha_inicio': '2026-01-01', 'fecha_fin': '2026-02-28', 'run_id': 'manual_web_revision_01'}
INFO: Datos cargados correctamente: 120 registros encontrados.
INFO: Se descartaron 1 registros por fechas inválidas.
INFO: Se descartaron 3 registros con unidades <= 0.
INFO: Se imputaron 4 precios faltantes usando la mediana del set (7500.0).
INFO: Filtro aplicado exitosamente. Quedan 13 registros.
INFO: Pipeline finalizado exitosamente. Artefactos guardados.


In [ ]:
# # PRUEBA - Verificar riesgo de sobrescritura
# # No crea, modifica ni elimina archivos.

# artefactos = [
#     "detalle.csv",
#     "resumen.json",
#     "entorno.json",
#     "ejecucion.log"
# ]

# for nombre in artefactos:
#     ruta = RUN_DIR / nombre

#     print(
#         f"{nombre}: "
#         f"{'EXISTE' if ruta.is_file() else 'NO EXISTE'}"
#     )

detalle.csv: EXISTE
resumen.json: EXISTE
entorno.json: EXISTE
ejecucion.log: EXISTE


In [ ]:
# ### Guardo los originales
# import hashlib

# ruta_protegida = RAIZ / directorio_salidas / "manual_web"

# archivos = [
#     "detalle.csv",
#     "resumen.json",
#     "entorno.json",
#     "ejecucion.log"
# ]

# def hash_archivo(ruta):
#     return hashlib.sha256(ruta.read_bytes()).hexdigest()

# hashes_antes = {
#     nombre: hash_archivo(ruta_protegida / nombre)
#     for nombre in archivos
# }

# print("PRUEBA 1 OK: Huellas originales capturadas.")

PRUEBA 1 OK: Huellas originales capturadas.


In [ ]:
# hashes_despues = {
#     nombre: hash_archivo(ruta_protegida / nombre)
#     for nombre in archivos
# }

# for nombre in archivos:
#     assert hashes_antes[nombre] == hashes_despues[nombre], (
#         f"ERROR: El archivo {nombre} fue modificado."
#     )

#     print(f"OK: {nombre} permanece intacto.")

# print("\nPRUEBA NEGATIVA SATISFACTORIA.")

OK: detalle.csv permanece intacto.
OK: resumen.json permanece intacto.
OK: entorno.json permanece intacto.
OK: ejecucion.log permanece intacto.

PRUEBA NEGATIVA SATISFACTORIA.


In [ ]:
# # PRUEBA FINAL - Protección de artefactos entre corridas

# from pathlib import Path
# import hashlib

# artefactos = [
#     "detalle.csv",
#     "resumen.json",
#     "entorno.json",
#     "ejecucion.log"
# ]

# def calcular_hash(ruta: Path) -> str:
#     return hashlib.sha256(ruta.read_bytes()).hexdigest()

# # Directorio de la ejecución anterior
# ruta_anterior = RAIZ / directorio_salidas / "manual_web"

# # Directorio de la nueva ejecución
# ruta_nueva = RAIZ / directorio_salidas / "manual_web_revision_01"

# # PRUEBA 1 - Los cuatro artefactos nuevos existen
# for nombre in artefactos:
#     archivo = ruta_nueva / nombre

#     assert archivo.is_file(), (
#         f"ERROR: No existe el nuevo artefacto {nombre}."
#     )

#     assert archivo.stat().st_size > 0, (
#         f"ERROR: El nuevo artefacto {nombre} está vacío."
#     )

# print("PRUEBA 1 OK: Los cuatro artefactos nuevos existen.")

# # PRUEBA 2 - Los anteriores permanecen intactos
# for nombre in artefactos:
#     hash_actual = calcular_hash(ruta_anterior / nombre)

#     assert hash_actual == hashes_antes[nombre], (
#         f"ERROR: Se modificó el artefacto anterior {nombre}."
#     )

# print("PRUEBA 2 OK: Los cuatro artefactos anteriores están intactos.")

# print("\nPROTECCIÓN CONTRA SOBRESCRITURAS VALIDADA.")

PRUEBA 1 OK: Los cuatro artefactos nuevos existen.
PRUEBA 2 OK: Los cuatro artefactos anteriores están intactos.

PROTECCIÓN CONTRA SOBRESCRITURAS VALIDADA.


### Punto de control de calidad

Automatizá invariantes del resultado; no te limites a inspeccionarlo visualmente.


In [54]:
# TODO · Escribí las comprobaciones finales del reto.
# Deben fallar con mensajes útiles cuando:
# - el resultado esté vacío;
# - haya fechas inválidas o fuera del rango;
# - existan unidades no positivas o precios faltantes;
# - aparezca un canal distinto del solicitado;
# - el total del detalle no coincida con el resumen;
# - falte alguno de los cuatro artefactos esperados.


# ============================================================
# PUNTO DE CONTROL DE CALIDAD
# Comprobaciones automáticas de la corrida actual
# ============================================================

# PRUEBA 1 - Verificar que el resultado no esté vacío

assert isinstance(detalle, pd.DataFrame), (
    "ERROR: detalle debe ser un DataFrame."
)

assert not detalle.empty, (
    "ERROR: El resultado está vacío. Revisar filtros."
)

print("PRUEBA 1 OK: Resultado no vacío.")


# PRUEBA 2 - Verificar fechas válidas y dentro del rango

fechas = pd.to_datetime(detalle["fecha"], errors="coerce")

assert fechas.notna().all(), (
    "ERROR: Existen fechas inválidas o nulas."
)

inicio_dt = pd.to_datetime(fecha_inicio)
fin_dt = pd.to_datetime(fecha_fin)

assert inicio_dt <= fin_dt, (
    "ERROR: El período solicitado está invertido."
)

assert fechas.between(inicio_dt, fin_dt).all(), (
    "ERROR: Existen fechas fuera del período solicitado."
)

print("PRUEBA 2 OK: Fechas válidas y dentro del rango.")


# PRUEBA 3 - Verificar unidades positivas

assert (detalle["unidades"] > 0).all(), (
    "ERROR: Existen unidades menores o iguales a cero."
)

print("PRUEBA 3 OK: Todas las unidades son positivas.")


# PRUEBA 4 - Verificar precios sin nulos

assert detalle["precio_unitario"].notna().all(), (
    "ERROR: Existen precios faltantes."
)

print("PRUEBA 4 OK: No existen precios faltantes.")


# PRUEBA 5 - Verificar el canal solicitado

assert detalle["canal"].eq(canal).all(), (
    f"ERROR: Existen canales diferentes de '{canal}'."
)

print("PRUEBA 5 OK: Canal correcto.")


# PRUEBA 6 - Verificar coherencia del resumen

assert resumen["registros_finales"] == len(detalle), (
    "ERROR: Cantidad de registros inconsistente."
)

total_detalle = float(detalle["venta_total"].sum())
total_resumen = float(resumen["venta_total"])

assert abs(total_detalle - total_resumen) <= 0.01, (
    f"ERROR: Total inconsistente. "
    f"Detalle={total_detalle}, Resumen={total_resumen}."
)

assert resumen["parametros"] == PARAMETROS, (
    "ERROR: Los parámetros del resumen no coinciden."
)

print("PRUEBA 6 OK: Resumen coherente con el detalle.")


# PRUEBA 7 - Verificar los cuatro artefactos

artefactos = [
    "detalle.csv",
    "resumen.json",
    "entorno.json",
    "ejecucion.log"
]

for nombre in artefactos:
    archivo = RUN_DIR / nombre

    assert archivo.is_file(), (
        f"ERROR: Falta el artefacto {nombre}."
    )

    assert archivo.stat().st_size > 0, (
        f"ERROR: El artefacto {nombre} está vacío."
    )

print("PRUEBA 7 OK: Cuatro artefactos presentes.")


# ============================================================
# RESULTADO FINAL
# ============================================================

print("\n============================================")
print("PUNTO DE CONTROL DE CALIDAD: SATISFACTORIO")
print("============================================")
print("Run ID:", run_id)
print("Registros finales:", len(detalle))
print("Venta total:", total_detalle)
print("Directorio:", RUN_DIR)



PRUEBA 1 OK: Resultado no vacío.
PRUEBA 2 OK: Fechas válidas y dentro del rango.
PRUEBA 3 OK: Todas las unidades son positivas.
PRUEBA 4 OK: No existen precios faltantes.
PRUEBA 5 OK: Canal correcto.
PRUEBA 6 OK: Resumen coherente con el detalle.
PRUEBA 7 OK: Cuatro artefactos presentes.

PUNTO DE CONTROL DE CALIDAD: SATISFACTORIO
Run ID: manual_web_revision_01
Registros finales: 13
Venta total: 382500.0
Directorio: c:\codigo\python\ambientes\ProgramacionAvanzada_Reto-Modulo-4\resultados\reto_integrador\manual_web_revision_01


## Hito 4 · Ejecutar el mismo notebook dos veces

Guardá primero este notebook. Desde una terminal ubicada en `taller_integrador`, reemplazá `TU_NOTEBOOK.ipynb` por su nombre y ejecutá:

```bash
papermill TU_NOTEBOOK.ipynb ejecutado_web.ipynb -k unicaba-mod4 \
  -p canal Web -p fecha_inicio 2026-01-01 -p fecha_fin 2026-02-28 -p run_id web_bimestre

papermill TU_NOTEBOOK.ipynb ejecutado_tienda.ipynb -k unicaba-mod4 \
  -p canal Tienda -p fecha_inicio 2026-03-01 -p fecha_fin 2026-04-30 -p run_id tienda_bimestre
```

> En PowerShell, escribí cada comando en una sola línea. Los dos notebooks ejecutados quedan en `taller_integrador`; los cuatro artefactos de cada corrida quedan bajo `resultados/reto_integrador/<run_id>/`.

### Evidencia a completar

| Corrida | Parámetros | Filas | Venta total | Advertencias relevantes | Ruta de artefactos |
|---|---|---:|---:|---|---|
| `web_bimestre` | Canal: Web; inicio: 2026-01-01; fin: 2026-02-28 | 13 | 382500.0 | Normalización de 1 registro `Wb` a `Web`; descarte de 1 fecha inválida y 3 registros con unidades <= 0; imputación de 4 precios faltantes mediante mediana de 7500.0. | `resultados/reto_integrador/web_bimestre/` |
| `tienda_bimestre` | Canal: Tienda; inicio: 2026-03-01; fin: 2026-04-30 | 14 | 470000.0 | Normalización de 1 registro `Wb` a `Web`; descarte de 1 fecha inválida y 3 registros con unidades <= 0; imputación de 4 precios faltantes mediante mediana de 7500.0. | `resultados/reto_integrador/tienda_bimestre/` |


### Pregunta

**¿Qué parte de la diferencia entre resultados se explica por parámetros y qué parte por reglas de calidad? Respondé con evidencia del JSON y del log.**

**1. Diferencias explicadas por los parámetros**

Los archivos `resumen.json` evidencian que ambas corridas utilizaron diferentes parámetros de canal y período:

- **`web_bimestre`:** canal Web, del 01/01/2026 al 28/02/2026. Se obtuvieron 13 registros finales y una venta total de 382500.0.
- **`tienda_bimestre`:** canal Tienda, del 01/03/2026 al 30/04/2026. Se obtuvieron 14 registros finales y una venta total de 470000.0.

La diferencia observada es de **1 registro y 87500.0 unidades monetarias** a favor de la corrida Tienda.

Esta diferencia se observa al seleccionar distintos subconjuntos del dataset procesado. Sin embargo, no puede atribuirse exclusivamente al canal, dado que también cambia el período de análisis.

**2. Efecto de las reglas de calidad**

Los archivos `ejecucion.log` de ambas corridas registran las mismas operaciones de limpieza:

- Normalización de 1 registro con canal `Wb` a `Web` (`WARNING`).
- Descarte de 1 registro por fecha inválida (`INFO`).
- Descarte de 3 registros con unidades menores o iguales a cero (`INFO`).
- Imputación de 4 precios faltantes utilizando una mediana de 7500.0 (`INFO`).

Estas reglas se aplicaron sobre los 120 registros originales **antes de filtrar por canal y período**, conservándose 116 registros después de los descartes.

Por lo tanto, ambas ejecuciones parten de un conjunto de datos depurado mediante las mismas reglas. Posteriormente, cada filtro selecciona los registros correspondientes a sus parámetros.

Los logs permiten identificar las transformaciones efectuadas, pero no cuantificar directamente cuánto de la diferencia de 87500.0 se debe a cada regla de calidad. Para calcular ese impacto sería necesario analizar específicamente los registros afectados y comparar los resultados bajo condiciones controladas.

**3. Conclusión**

La evidencia de `resumen.json` y `ejecucion.log` demuestra que las dos corridas aplicaron consistentemente las mismas reglas de calidad, pero obtuvieron resultados diferentes al utilizar distintos parámetros de filtrado.

La diferencia observada entre los resultados finales corresponde a la comparación de dos subconjuntos distintos del dataset depurado. No constituye una medida aislada del efecto del canal ni permite determinar, por sí sola, el impacto monetario individual de las reglas de limpieza.


## Hito 5 · Contraste crítico con IA

### Antes de consultar
Escribí tu diagnóstico humano en 3–5 líneas: ¿cuál es el mayor riesgo de reproducibilidad u observabilidad que todavía queda en tu solución?

**Mi diagnóstico:**

> …Diagnostico

El mayor riesgo de reproducibilidad pendiente es la modificación o pérdida del dataset original `ventas_ejemplo.csv`, del cual depende el procesamiento. Aunque registramos su hash SHA-256 en `entorno.json`, este mecanismo permite detectar cambios, pero no prevenirlos ni recuperar el contenido original. Si el archivo cambia y no conservamos una copia de la versión utilizada, no podremos garantizar la reproducción exacta de los resultados anteriores. Una posible mejora sería incorporar versionado o almacenamiento inmutable de los datos de entrada.

### Consulta controlada
Usá el mismo notebook y el mismo prompt con **dos modelos distintos**, si están disponibles. Si sólo tenés acceso a uno, comparalo con tu diagnóstico humano: esa alternativa no impide completar el reto.

No compartas credenciales, rutas personales ni datos no autorizados. Podés usar este prompt:

```text
Actuá como revisor de reproducibilidad de un pipeline de datos. Analizá el notebook adjunto sin reescribirlo completo. Identificá un único riesgo de alto impacto relacionado con entorno, parámetros, logging, orden de ejecución o artefactos. Respondé con: (1) evidencia concreta, (2) causa probable, (3) cambio mínimo, (4) prueba que demostraría la corrección. Si falta evidencia, decilo explícitamente.
```

### Registro de la consulta

Para cada consulta anotá: herramienta, modelo, fecha, `run_id` analizado, SHA-256 de los datos y si adjuntaste exactamente la misma versión del notebook. Conservá la respuesta completa como anexo o pegá un extracto identificable.


### Matriz de contraste

| Fuente / modelo / fecha | Entrada o `run_id` | Riesgo identificado | Evidencia citada | Cambio mínimo | Prueba ejecutada | ¿Acepto? ¿Por qué? |
|---|---|---|---|---|---|---|
| Gemini — 15/10/2026 | Notebook revisado inicialmente; corrida `manual_web` | Posible contaminación temporal o entre canales al calcular la mediana de precios sobre todo el dataset antes de aplicar el filtro. | En la orquestación, `limpiar_ventas()` se ejecuta antes de `filtrar_ventas()`. La imputación utiliza la mediana global de precios. | Evaluar una imputación por subconjunto de análisis, conservando previamente la normalización de canales y la validación de fechas. No invertir simplemente las funciones, porque el filtro depende de esas transformaciones. | No validada experimentalmente. Se propuso agregar un registro futuro con precio elevado, pero esa prueba no garantiza por sí sola una variación de la mediana. | **Acepto el riesgo como hipótesis para investigar, pero no la corrección propuesta sin pruebas.** La mediana global puede incorporar información de otros períodos o canales. Debe verificarse si eso contradice los requisitos del análisis y cuál sería su impacto. |
| Diagnóstico humano — 15/10/2026 | Dataset `ventas_ejemplo.csv` y código de carga mediante `pd.read_csv()` | Dependencia de un archivo local modificable. Si cambia el CSV original, podrían cambiar los resultados y perderse la posibilidad de reproducir una ejecución histórica. | `cargar_ventas()` lee el CSV local. `capturar_entorno()` registra su SHA-256, pero actualmente no lo compara contra un hash esperado ni conserva una copia inmutable. | Validar el SHA-256 contra una referencia confiable antes del procesamiento y conservar una versión identificable del dataset utilizado. | Según el registro de la revisión, se modificó manualmente un dato y se observó que el procesamiento aceptaba el archivo alterado. No se adjunta en esta matriz una comparación verificable de hashes anterior y posterior. | **Acepto el diagnóstico.** El hash registrado permite comprobar la identidad de los datos, pero no impide alteraciones ni permite recuperarlos. En producción deben existir controles de integridad, permisos y versionado de la fuente. |
| ChatGPT — GPT-6 — consulta del 15/10/2026; validación del 18/10/2026 | Notebook del Reto Integrador; `run_id=manual_web` | Sobrescritura de logs y artefactos cuando se reutiliza un mismo `run_id`, con pérdida de trazabilidad de ejecuciones anteriores. | `RUN_DIR.mkdir(parents=True, exist_ok=True)` permitía reutilizar el directorio; `FileHandler(mode="w")`, `to_csv()` y la apertura de archivos JSON con `"w"` podían reemplazar su contenido. | Utilizar `RUN_DIR.mkdir(parents=True, exist_ok=False)` y rechazar un `run_id` existente mediante `FileExistsError`, antes de inicializar el logger. | **Ejecutada y satisfactoria.** Se intentó reutilizar `manual_web`, se obtuvo `FileExistsError` y se verificó mediante SHA-256 que los cuatro artefactos anteriores permanecían intactos. Posteriormente, una corrida con `manual_web_revision_01` finalizó correctamente. | **Sí, acepto e implementé la corrección.** Las pruebas positivas y negativas respaldan su eficacia para impedir la reutilización accidental del directorio. Se documenta como limitación que `Restart & Run All` falla si el `run_id` ya existe. |

### Evaluación crítica de las recomendaciones

Las tres revisiones identificaron riesgos diferentes, pero no todas tienen el mismo nivel de validación.

La recomendación de Gemini plantea un posible problema de alcance temporal y segmentación de la imputación. Sin embargo, modificar directamente el orden de las funciones podría romper el procesamiento, dado que el filtrado requiere datos previamente normalizados. La propuesta queda pendiente de una prueba controlada.

El diagnóstico humano identificó una debilidad en la preservación de los datos originales. El SHA-256 constituye evidencia de identidad e integridad verificable, pero no reemplaza al versionado ni al almacenamiento protegido.

La recomendación de ChatGPT sobre sobrescrituras fue implementada y validada mediante pruebas de rechazo, generación de una corrida nueva y comparación de hashes SHA-256. Por ello, es el único cambio de estas tres revisiones cuya corrección quedó demostrada mediante las pruebas específicas registradas.

**Conclusión:** las sugerencias de IA deben tratarse como hipótesis técnicas sujetas a verificación. Una propuesta plausible no debe incorporarse al pipeline sin analizar sus dependencias, probar su comportamiento y comprobar que no introduce nuevas limitaciones.

### Limitaciones del contraste

Las consultas consignadas corresponden a revisiones realizadas en distintos momentos. Para considerarlas una comparación controlada entre modelos, sería necesario acreditar que recibieron exactamente la misma versión del notebook, el mismo prompt y la misma evidencia.

También corresponde conservar las respuestas originales y registrar el modelo, la fecha, el `run_id` y el SHA-256 del dataset utilizado en cada consulta. Estos datos no deben completarse retrospectivamente sin evidencia.



## Cierre · Decisión técnica (250–350 palabras)

Redactá un informe breve que responda:

1. ¿Qué hacía frágil al proceso inicial?
2. ¿Qué evidencia permite ahora reconstruir cada corrida?
3. ¿Qué diferencia encontraste entre las dos ejecuciones parametrizadas?
4. ¿Qué sugerencia humana o de IA aceptaste o rechazaste, y con qué prueba?
5. ¿Qué riesgo queda pendiente si este proceso pasara a producción?

## Cierre · Decisión técnica

El proceso inicial era frágil porque dependía de ejecuciones manuales, un entorno local y archivos de entrada cuyo estado no quedaba suficientemente documentado. Sin parámetros explícitos, registros de limpieza y artefactos separados, resultaba difícil explicar por qué cambiaba un informe o reproducir una corrida anterior.

La solución incorporó un entorno definido mediante `requirements.txt`, parámetros de canal, fechas y `run_id`, y un pipeline que registra sus transformaciones. Cada ejecución genera `detalle.csv`, `resumen.json`, `entorno.json` y `ejecucion.log`. El recibo de entorno incluye versiones y hashes SHA-256 del CSV y de las dependencias; el log registra normalizaciones, descartes, imputaciones y cantidad final de registros. Además, las siete comprobaciones del punto de control de calidad finalizaron satisfactoriamente.

Papermill completó 39 de 39 celdas en cada configuración. Según los respectivos `resumen.json`, `web_bimestre` (Web, enero-febrero) produjo 13 registros y 382.500 de venta total, mientras que `tienda_bimestre` (Tienda, marzo-abril) produjo 14 registros y 470.000. La diferencia de 87.500 no puede atribuirse únicamente al canal porque también cambió el período. Ambos logs muestran las mismas reglas de limpieza global: una normalización de `Wb` a `Web`, cuatro registros descartados y cuatro precios imputados.

Acepté la recomendación de impedir la reutilización accidental de un `run_id`: cambié la creación del directorio a `exist_ok=False`. Una prueba negativa confirmó `FileExistsError` y comprobó mediante SHA-256 que los cuatro artefactos anteriores no se modificaron; una corrida con identificador nuevo se completó correctamente. En cambio, dejé pendiente la propuesta de Gemini de cambiar el orden de limpieza y filtrado, porque esa modificación directa rompería dependencias del pipeline y carece de una prueba concluyente.

Para producción persisten dos riesgos: el hash permite detectar cambios del CSV, pero no recuperar una versión perdida, y la política de rechazo requiere gestionar nuevos identificadores para las reejecuciones, incluido «Reiniciar kernel y ejecutar todo». Harían falta versionado de entradas y una política operativa de reejecución.

### Lista de entrega

- [ ] El entorno se recrea desde `requirements.txt`.
- [ ] `pip check` pasa y las huellas/resultado coinciden en el entorno limpio.
- [ ] El notebook supera **Reiniciar kernel y ejecutar todo**.
- [ ] Se conservan dos notebooks ejecutados con Papermill, uno por configuración.
- [ ] Cada `run_id` conserva sus cuatro artefactos.
- [ ] El log permite reconstruir parámetros, advertencias y resultado.
- [ ] Las comprobaciones automáticas pasan.
- [ ] La matriz de IA distingue sugerencia de evidencia.
- [ ] La decisión final tiene 250–350 palabras.

### Mini-rúbrica del cierre

- **Evidencia:** cada afirmación importante cita un log, JSON o comprobación concreta.
- **Juicio sobre IA:** la aceptación o rechazo de una sugerencia incluye una prueba ejecutada.
- **Explicación:** distingue cambios causados por parámetros de cambios causados por reglas de calidad.

## Extensión opcional
Agregá un parámetro `politica_precios` (`mediana` o `descartar`) y compará cómo cambia el resultado sin duplicar el notebook.
